# MAPA - Business Success Predictor

## Phase 2: Machine Learning Model

This notebook trains a model to predict whether a new business will be successful based on its location features.

**Success criteria:** Business achieves 4.1+ stars AND has 100+ reviews on Google.

**Features:** Mostly binary (has_parking, near_school, on_main_road, etc.) plus some numeric (review count, nearby amenity counts, competitor density).

**Models:** Logistic Regression, Random Forest, XGBoost -- we compare all three.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    f1_score,
)
import joblib
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All imports loaded successfully.")

## 2. Load Data from Phase 1

Load JSON files saved by the Streamlit dashboard. Each file contains businesses, reviews, analysis, and nearby context for one search.

In [ ]:
DATA_DIR = os.path.join("..", "data")

json_files = glob.glob(os.path.join(DATA_DIR, "*.json"))
print(f"Found {len(json_files)} data files:")
for f in json_files:
    print(f"  - {os.path.basename(f)}")

In [ ]:
all_businesses = []

for filepath in json_files:
    with open(filepath, "r") as f:
        data = json.load(f)

    businesses = data.get("businesses", [])
    nearby = data.get("nearby", {})

    for biz in businesses:
        row = {
            "name": biz.get("name", ""),
            "rating": biz.get("rating"),
            "total_reviews": biz.get("total_reviews", 0),
            "lat": biz.get("lat"),
            "lng": biz.get("lng"),
        }

        biz_name = biz.get("name", "")
        if biz_name in nearby:
            biz_nearby = nearby[biz_name]
            for category, cat_data in biz_nearby.items():
                row[f"has_{category}"] = 1 if cat_data.get("has_nearby") else 0
                row[f"{category}_count"] = cat_data.get("count", 0)
                row[f"{category}_closest_km"] = cat_data.get("closest_distance_km")
        else:
            for category in ["parking", "public_transport", "schools", "main_road",
                             "shopping", "hospital", "park", "tourist_attraction"]:
                row[f"has_{category}"] = 0
                row[f"{category}_count"] = 0
                row[f"{category}_closest_km"] = None

        all_businesses.append(row)

df = pd.DataFrame(all_businesses)
print(f"Total businesses loaded: {len(df)}")
df.head()

## 3. Data Preparation

Create the target variable (success = rating >= 4.1 AND total_reviews >= 100) and prepare features.

In [ ]:
SUCCESS_RATING = 4.1
SUCCESS_REVIEWS = 100

df = df.dropna(subset=["rating"])

df["success"] = ((df["rating"] >= SUCCESS_RATING) & (df["total_reviews"] >= SUCCESS_REVIEWS)).astype(int)

print(f"Total businesses with ratings: {len(df)}")
print(f"\nSuccess criteria: rating >= {SUCCESS_RATING} AND total_reviews >= {SUCCESS_REVIEWS}")
print(f"\nSuccess distribution:")
print(df["success"].value_counts())
print(f"\nSuccess rate: {df['success'].mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df["rating"], bins=20, color="#667eea", edgecolor="white")
axes[0].axvline(x=SUCCESS_RATING, color="red", linestyle="--", label=f"Rating threshold: {SUCCESS_RATING}")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Rating Distribution")
axes[0].legend()

axes[1].hist(df["total_reviews"], bins=30, color="#764ba2", edgecolor="white")
axes[1].axvline(x=SUCCESS_REVIEWS, color="red", linestyle="--", label=f"Review threshold: {SUCCESS_REVIEWS}")
axes[1].set_xlabel("Total Reviews")
axes[1].set_ylabel("Count")
axes[1].set_title("Review Count Distribution")
axes[1].legend()

success_counts = df["success"].value_counts()
labels = ["Not Successful", "Successful"]
axes[2].pie(
    success_counts.values,
    labels=labels,
    colors=["#dc3545", "#28a745"],
    autopct="%1.1f%%",
    startangle=90,
)
axes[2].set_title("Success vs Non-Success\n(4.1+ stars AND 100+ reviews)")

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
binary_features = [
    "has_parking",
    "has_public_transport",
    "has_schools",
    "has_main_road",
    "has_shopping",
    "has_hospital",
    "has_park",
    "has_tourist_attraction",
]

count_features = [
    "parking_count",
    "public_transport_count",
    "schools_count",
    "main_road_count",
    "shopping_count",
    "hospital_count",
    "park_count",
    "tourist_attraction_count",
]

distance_features = [
    "parking_closest_km",
    "public_transport_closest_km",
    "schools_closest_km",
    "main_road_closest_km",
    "shopping_closest_km",
    "hospital_closest_km",
    "park_closest_km",
    "tourist_attraction_closest_km",
]

numeric_features = ["total_reviews"]

all_features = binary_features + count_features + distance_features + numeric_features

available_features = [f for f in all_features if f in df.columns]
print(f"Available features: {len(available_features)}")
for f in available_features:
    print(f"  - {f}")

In [ ]:
for col in available_features:
    if col in distance_features:
        df[col] = df[col].fillna(999)
    else:
        df[col] = df[col].fillna(0)

df["total_nearby_amenities"] = df[[c for c in count_features if c in df.columns]].sum(axis=1)
df["amenity_diversity"] = df[[c for c in binary_features if c in df.columns]].sum(axis=1)

available_features.extend(["total_nearby_amenities", "amenity_diversity"])

print(f"\nFinal feature count: {len(available_features)}")
print(f"\nFeature summary:")
df[available_features].describe().round(2)

## 5. Feature Correlation Analysis

In [ ]:
corr_features = available_features + ["success"]
corr_matrix = df[corr_features].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
success_corr = corr_matrix["success"].drop("success").sort_values(ascending=False)
print("Feature correlation with success:")
print("=" * 50)
for feat, corr in success_corr.items():
    direction = "+" if corr > 0 else "-"
    print(f"  {direction} {feat}: {corr:.4f}")

## 6. Train/Test Split

In [ ]:
X = df[available_features]
y = df["success"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining success rate: {y_train.mean():.2%}")
print(f"Test success rate: {y_test.mean():.2%}")

## 7. Model Training and Comparison

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42, class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False,
    ),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n{'=' * 50}")
    print(f"Training: {name}")
    print(f"{'=' * 50}")

    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring="f1")
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        "model": model,
        "accuracy": acc,
        "f1_score": f1,
        "auc_roc": auc,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC:  {auc:.4f}")
    print(f"CV F1:    {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Below 4.1", "4.1+ Stars"]))

## 8. Model Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = list(results.keys())
metrics_data = {
    "Accuracy": [results[m]["accuracy"] for m in model_names],
    "F1 Score": [results[m]["f1_score"] for m in model_names],
    "AUC-ROC": [results[m]["auc_roc"] for m in model_names],
}

colors = ["#667eea", "#764ba2", "#f093fb"]

for idx, (metric_name, values) in enumerate(metrics_data.items()):
    bars = axes[idx].bar(model_names, values, color=colors)
    axes[idx].set_title(metric_name)
    axes[idx].set_ylim(0, 1)
    for bar, val in zip(bars, values):
        axes[idx].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.3f}",
            ha="center",
            fontsize=11,
        )

plt.suptitle("Model Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc_roc']:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Below 4.1", "4.1+"],
        yticklabels=["Below 4.1", "4.1+"],
        ax=axes[idx],
    )
    axes[idx].set_title(f"{name}")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xlabel("Predicted")

plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
best_model_name = max(results, key=lambda k: results[k]["f1_score"])
best_model = results[best_model_name]["model"]
print(f"Best model: {best_model_name} (F1 = {results[best_model_name]['f1_score']:.4f})")

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
else:
    importances = np.zeros(len(available_features))

feat_imp = pd.DataFrame({
    "Feature": available_features,
    "Importance": importances,
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feat_imp["Feature"], feat_imp["Importance"], color="#667eea")
plt.xlabel("Importance")
plt.title(f"Feature Importance ({best_model_name})")
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

## 10. Save the Best Model

In [ ]:
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "best_model.joblib")
scaler_path = os.path.join(MODEL_DIR, "scaler.joblib")
features_path = os.path.join(MODEL_DIR, "features.json")

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)

with open(features_path, "w") as f:
    json.dump({
        "features": available_features,
        "binary_features": binary_features,
        "count_features": count_features,
        "distance_features": distance_features,
        "numeric_features": numeric_features,
        "best_model": best_model_name,
        "success_rating_threshold": SUCCESS_RATING,
        "success_review_threshold": SUCCESS_REVIEWS,
    }, f, indent=2)

print(f"Model saved to: {model_path}")
print(f"Scaler saved to: {scaler_path}")
print(f"Feature list saved to: {features_path}")
print(f"\nBest model: {best_model_name}")
print(f"Success criteria: {SUCCESS_RATING}+ stars AND {SUCCESS_REVIEWS}+ reviews")
print(f"F1 Score: {results[best_model_name]['f1_score']:.4f}")
print(f"AUC-ROC: {results[best_model_name]['auc_roc']:.4f}")

## 11. Prediction Function

This function will be used in the Streamlit dashboard to predict success for a new business.

In [ ]:
def predict_success(features_dict, model_path="../models/best_model.joblib",
                    scaler_path="../models/scaler.joblib",
                    features_path="../models/features.json"):
    """
    Predict whether a new business will be successful.

    Parameters:
        features_dict: dict with feature names as keys and values
        Example: {"has_parking": 1, "has_schools": 0, "total_reviews": 50, ...}

    Returns:
        dict with prediction, probability, and feature contributions
    """
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)

    with open(features_path, "r") as f:
        feature_config = json.load(f)

    feature_names = feature_config["features"]
    model_name = feature_config["best_model"]

    feature_values = [features_dict.get(f, 0) for f in feature_names]
    X_input = pd.DataFrame([feature_values], columns=feature_names)

    if model_name == "Logistic Regression":
        X_input = scaler.transform(X_input)

    prediction = model.predict(X_input)[0]
    probability = model.predict_proba(X_input)[0]

    return {
        "success": bool(prediction),
        "probability_success": float(probability[1]),
        "probability_failure": float(probability[0]),
        "model_used": model_name,
        "threshold": feature_config["success_threshold"],
    }


print("Prediction function defined. Ready for use in the dashboard.")

In [ ]:
test_input = {
    "has_parking": 1,
    "has_public_transport": 1,
    "has_schools": 1,
    "has_main_road": 1,
    "has_shopping": 0,
    "has_hospital": 0,
    "has_park": 1,
    "has_tourist_attraction": 0,
    "parking_count": 3,
    "public_transport_count": 2,
    "schools_count": 1,
    "main_road_count": 1,
    "shopping_count": 0,
    "hospital_count": 0,
    "park_count": 1,
    "tourist_attraction_count": 0,
    "parking_closest_km": 0.1,
    "public_transport_closest_km": 0.2,
    "schools_closest_km": 0.3,
    "main_road_closest_km": 0.05,
    "shopping_closest_km": 999,
    "hospital_closest_km": 999,
    "park_closest_km": 0.15,
    "tourist_attraction_closest_km": 999,
    "total_reviews": 0,
    "total_nearby_amenities": 8,
    "amenity_diversity": 5,
}

result = predict_success(test_input)
print(f"Prediction: {'SUCCESS' if result['success'] else 'UNLIKELY'}")
print(f"Success probability: {result['probability_success']:.2%}")
print(f"Model used: {result['model_used']}")